# Atividade: PPO (Proximal Policy Optimization) do zero com PyTorch 🤖


Neste notebook, **você implementará o PPO do zero com PyTorch** e treinará um agente para pousar uma nave lunar 🚀🌕.

Este notebook é uma simplificação do unit8 (parte 1) do **Curso de Aprendizagem por Reforço Profundo** (Deep Reinforcement Learning) https://huggingface.co/datasets/huggingface-deep-rl-course/, que por sua vez se baseia na implementação do [CleanRL](https://github.com/vwxyzjn/cleanrl).

> ⚠️ **Versão adaptada para as versões atuais**: o original usava gym 0.22 (descontinuado) e `distutils` (removido do Python). Aqui o código usa o **gymnasium ≥ 1.0** e funciona no Colab de hoje.


### 🎮 Ambiente:

- [LunarLander-v3](https://gymnasium.farama.org/environments/box2d/lunar_lander/) — pousar uma nave lunar com 4 ações discretas (nada, motor esquerdo, motor principal, motor direito).

### 📚 Biblioteca:

- [PyTorch](https://pytorch.org/) — o PPO é implementado do zero (código do CleanRL, sem biblioteca de RL pronta).

## Objetivos deste notebook 🏆
No final do notebook, você irá:
- Compreender o **PPO com clipping** e o cálculo de vantagens com **GAE** (Generalized Advantage Estimation).
- Implementar o algoritmo **do zero** com PyTorch (arquitetura ator-crítico).
- Treinar, avaliar e **gravar um vídeo** do agente pousando a nave.

## Instalar as dependências 🔽

O PyTorch já vem no Colab. Precisamos de:
- `gymnasium[box2d]`: o LunarLander (motor de física Box2D)
- `gymnasium[classic-control]`: pygame, para renderizar o ambiente no vídeo
- `imageio` + `imageio-ffmpeg`: gravação do vídeo de replay

In [29]:
%pip install -q "gymnasium[box2d]" "gymnasium[classic-control]" imageio imageio-ffmpeg

## O código do PPO 🔥

A célula a seguir define os **hiperparâmetros** (via `argparse`, como no CleanRL — você pode mexer nos valores padrão). Em seguida, a célula do **agente** e a célula do **loop de treino**.

O PPO combina:
- **Ator-crítico**: uma rede com duas cabeças — o ator (distribuição sobre as ações) e o crítico (valor do estado);
- **Clipping**: a razão `π_nova/π_velha` é limitada ao intervalo `[1−ε, 1+ε]`, evitando atualizações grandes demais;
- **GAE**: as vantagens combinam várias estimativas de 1 passo com desconto `γ·λ`, reduzindo variância;
- **K épocas de atualização** por lote, com normalização das vantagens.

In [30]:
import argparse
import os
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions.categorical import Categorical
from torch.utils.tensorboard import SummaryWriter

import gymnasium as gym


def strtobool(x):
    # substitui o distutils.util.strtobool (removido no Python 3.12+)
    return str(x).lower() in ("true", "1", "yes", "y")


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--exp-name", type=str, default="ppo", help="nome deste experimento")
    parser.add_argument("--seed", type=int, default=1, help="semente do experimento")
    parser.add_argument("--torch-deterministic", type=strtobool, default=True, nargs="?", const=True)
    parser.add_argument("--cuda", type=strtobool, default=True, nargs="?", const=True,
                        help="usa GPU se disponível")
    parser.add_argument("--env-id", type=str, default="LunarLander-v3", help="id do ambiente")
    parser.add_argument("--total-timesteps", type=int, default=100_000,
                        help="total de timesteps do treino")
    parser.add_argument("--learning-rate", type=float, default=2.5e-4, help="taxa de aprendizado")
    parser.add_argument("--num-envs", type=int, default=4, help="ambientes em paralelo")
    parser.add_argument("--num-steps", type=int, default=128,
                        help="passos por ambiente em cada coleta")
    parser.add_argument("--anneal-lr", type=strtobool, default=True, nargs="?", const=True,
                        help="decai a taxa de aprendizado ao longo do treino")
    parser.add_argument("--gae", type=strtobool, default=True, nargs="?", const=True,
                        help="usa GAE para as vantagens")
    parser.add_argument("--gamma", type=float, default=0.99, help="fator de desconto")
    parser.add_argument("--gae-lambda", type=float, default=0.95, help="lambda do GAE")
    parser.add_argument("--num-minibatches", type=int, default=4, help="número de minibatches")
    parser.add_argument("--update-epochs", type=int, default=4,
                        help="épocas K de atualização por lote")
    parser.add_argument("--norm-adv", type=strtobool, default=True, nargs="?", const=True,
                        help="normaliza as vantagens")
    parser.add_argument("--clip-coef", type=float, default=0.2, help="coeficiente de clipping")
    parser.add_argument("--clip-vloss", type=strtobool, default=True, nargs="?", const=True,
                        help="clipping da perda do valor")
    parser.add_argument("--ent-coef", type=float, default=0.01, help="coeficiente da entropia")
    parser.add_argument("--vf-coef", type=float, default=0.5, help="coeficiente da função valor")
    parser.add_argument("--max-grad-norm", type=float, default=0.5,
                        help="norma máxima para o clipping do gradiente")
    parser.add_argument("--target-kl", type=float, default=None,
                        help="limiar de KL para parada antecipada")
    args = parser.parse_args([])  # lista vazia: no Jupyter/Colab, sys.argv contém
                                 # os argumentos do kernel (não os nossos)
    args.batch_size = int(args.num_envs * args.num_steps)
    args.minibatch_size = int(args.batch_size // args.num_minibatches)
    return args

In [31]:
def make_env(env_id, seed, idx):
    def thunk():
        env = gym.make(env_id)
        env = gym.wrappers.RecordEpisodeStatistics(env)
        env.reset(seed=seed)
        return env
    return thunk


def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer


class Agent(nn.Module):
    def __init__(self, envs):
        super().__init__()
        obs_dim = np.array(envs.single_observation_space.shape).prod()
        n_actions = envs.single_action_space.n
        # Crítico: estima V(s)
        self.critic = nn.Sequential(
            layer_init(nn.Linear(obs_dim, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 1), std=1.0),
        )
        # Ator: distribuição sobre as ações
        self.actor = nn.Sequential(
            layer_init(nn.Linear(obs_dim, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, n_actions), std=0.01),
        )

    def get_value(self, x):
        return self.critic(x)

    def get_action_and_value(self, x, action=None):
        logits = self.actor(x)
        probs = Categorical(logits=logits)
        if action is None:
            action = probs.sample()
        return action, probs.log_prob(action), probs.entropy(), self.critic(x)

In [32]:
def main(args):
    run_name = f"{args.env_id}__{args.exp_name}__{args.seed}__{int(time.time())}"
    writer = SummaryWriter(f"runs/{run_name}")
    writer.add_text(
        "hyperparameters",
        "|param|value|\n|-|-|\n%s" % ("\n".join([f"|{key}|{value}|" for key, value in vars(args).items()]),)
    )

    # Sementes (reprodutibilidade)
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    torch.backends.cudnn.deterministic = args.torch_deterministic

    device = torch.device("cuda" if torch.cuda.is_available() and args.cuda else "cpu")
    print("device:", device)

    # Cria os ambientes vetorizados
    envs = gym.vector.SyncVectorEnv(
        [make_env(args.env_id, args.seed + i, i) for i in range(args.num_envs)]
    )
    assert isinstance(envs.single_action_space, gym.spaces.Discrete), \
        "esta implementação aceita apenas espaços de ação discretos"

    agent = Agent(envs).to(device)
    optimizer = optim.Adam(agent.parameters(), lr=args.learning_rate, eps=1e-5)

    # Buffers de coleta
    obs = torch.zeros((args.num_steps, args.num_envs) + envs.single_observation_space.shape).to(device)
    actions = torch.zeros((args.num_steps, args.num_envs) + envs.single_action_space.shape).to(device)
    logprobs = torch.zeros((args.num_steps, args.num_envs)).to(device)
    rewards = torch.zeros((args.num_steps, args.num_envs)).to(device)
    dones = torch.zeros((args.num_steps, args.num_envs)).to(device)
    values = torch.zeros((args.num_steps, args.num_envs)).to(device)

    global_step = 0
    start_time = time.time()
    next_obs, _ = envs.reset(seed=args.seed)
    next_obs = torch.Tensor(next_obs).to(device)
    next_done = torch.zeros(args.num_envs).to(device)
    num_updates = args.total_timesteps // args.batch_size

    best_return = -float("inf")

    for update in range(1, num_updates + 1):
        # Annealing da taxa de aprendizado
        if args.anneal_lr:
            frac = 1.0 - (update - 1.0) / num_updates
            optimizer.param_groups[0]["lr"] = frac * args.learning_rate

        # Coleta: num_steps passos em cada ambiente
        for step in range(0, args.num_steps):
            global_step += 1 * args.num_envs
            obs[step] = next_obs
            dones[step] = next_done

            with torch.no_grad():
                action, logprob, _, value = agent.get_action_and_value(next_obs)
                values[step] = value.flatten()
            actions[step] = action
            logprobs[step] = logprob

            next_obs, reward, terminations, truncations, info = envs.step(action.cpu().numpy())
            done = np.logical_or(terminations, truncations)
            rewards[step] = torch.tensor(reward).to(device).view(-1)
            next_obs, next_done = torch.Tensor(next_obs).to(device), torch.Tensor(done).to(device)

            # no gymnasium >= 1.0, `info` é um dict que ganha a chave
            # "episode" (um valor por ambiente) quando algum episódio termina
            if "episode" in info:
                done_mask = np.logical_or(terminations, truncations)
                ep_ret = float(np.array(info["episode"]["r"])[done_mask][0])
                print(f"global_step={global_step}, retorno_do_episódio={ep_ret:.0f}")
                writer.add_scalar("charts/episodic_return", ep_ret, global_step)

                if ep_ret > best_return:
                    best_return = ep_ret
                    torch.save(agent.state_dict(), "ppo-lunarlander-best.pt")
                    print(f"Novo recorde! Modelo salvo em ppo-lunarlander.pt com retorno: {best_return:.0f} ✓")

        # Bootstrap do valor no fim da coleta
        with torch.no_grad():
            next_value = agent.get_value(next_obs).reshape(1, -1)
            if args.gae:
                advantages = torch.zeros_like(rewards).to(device)
                lastgaelam = 0
                for t in reversed(range(args.num_steps)):
                    if t == args.num_steps - 1:
                        nextnonterminal = 1.0 - next_done
                        nextvalues = next_value
                    else:
                        nextnonterminal = 1.0 - dones[t + 1]
                        nextvalues = values[t + 1]
                    delta = rewards[t] + args.gamma * nextvalues * nextnonterminal - values[t]
                    advantages[t] = lastgaelam = delta + args.gamma * args.gae_lambda * nextnonterminal * lastgaelam
                returns = advantages + values
            else:
                returns = torch.zeros_like(rewards).to(device)
                for t in reversed(range(args.num_steps)):
                    if t == args.num_steps - 1:
                        nextnonterminal = 1.0 - next_done
                        next_return = next_value
                    else:
                        nextnonterminal = 1.0 - dones[t + 1]
                        next_return = returns[t + 1]
                    returns[t] = rewards[t] + args.gamma * nextnonterminal * next_return
                advantages = returns - values

        # Achata o lote
        b_obs = obs.reshape((-1,) + envs.single_observation_space.shape)
        b_logprobs = logprobs.reshape(-1)
        b_actions = actions.reshape((-1,) + envs.single_action_space.shape)
        b_advantages = advantages.reshape(-1)
        b_returns = returns.reshape(-1)
        b_values = values.reshape(-1)

        # Atualiza o ator e o crítico
        b_inds = np.arange(args.batch_size)
        clipfracs = []
        for epoch in range(args.update_epochs):
            np.random.shuffle(b_inds)
            for start in range(0, args.batch_size, args.minibatch_size):
                end = start + args.minibatch_size
                mb_inds = b_inds[start:end]

                _, newlogprob, entropy, newvalue = agent.get_action_and_value(
                    b_obs[mb_inds], b_actions.long()[mb_inds])
                logratio = newlogprob - b_logprobs[mb_inds]
                ratio = logratio.exp()

                with torch.no_grad():
                    old_approx_kl = (-logratio).mean()
                    approx_kl = ((ratio - 1) - logratio).mean()
                    clipfracs += [((ratio - 1.0).abs() > args.clip_coef).float().mean().item()]

                mb_advantages = b_advantages[mb_inds]
                if args.norm_adv:
                    mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)

                # Perda da política (clipped surrogate)
                pg_loss1 = -mb_advantages * ratio
                pg_loss2 = -mb_advantages * torch.clamp(ratio, 1 - args.clip_coef, 1 + args.clip_coef)
                pg_loss = torch.max(pg_loss1, pg_loss2).mean()

                # Perda do valor (clipped)
                newvalue = newvalue.view(-1)
                if args.clip_vloss:
                    v_loss_unclipped = (newvalue - b_returns[mb_inds]) ** 2
                    v_clipped = b_values[mb_inds] + torch.clamp(
                        newvalue - b_values[mb_inds], -args.clip_coef, args.clip_coef)
                    v_loss_clipped = (v_clipped - b_returns[mb_inds]) ** 2
                    v_loss = 0.5 * torch.max(v_loss_unclipped, v_loss_clipped).mean()
                else:
                    v_loss = 0.5 * ((newvalue - b_returns[mb_inds]) ** 2).mean()

                entropy_loss = entropy.mean()
                loss = pg_loss - args.ent_coef * entropy_loss + v_loss * args.vf_coef

                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(agent.parameters(), args.max_grad_norm)
                optimizer.step()

            if args.target_kl is not None and approx_kl > args.target_kl:
                break

        y_pred, y_true = b_values.cpu().numpy(), b_returns.cpu().numpy()
        var_y = np.var(y_true)
        explained_var = np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y
        writer.add_scalar("charts/learning_rate", optimizer.param_groups[0]["lr"], global_step)
        writer.add_scalar("losses/value_loss", v_loss.item(), global_step)
        writer.add_scalar("losses/policy_loss", pg_loss.item(), global_step)
        writer.add_scalar("losses/entropy", entropy_loss.item(), global_step)
        writer.add_scalar("losses/approx_kl", approx_kl.item(), global_step)
        writer.add_scalar("losses/clipfrac", np.mean(clipfracs), global_step)
        writer.add_scalar("losses/explained_variance", explained_var, global_step)
        print(f"update={update}, SPS={int(global_step / (time.time() - start_time))}")

    envs.close()
    writer.close()

    # Garante salvar o modelo final
    torch.save(agent.state_dict(), "ppo-lunarlander-final.pt")
    print("Modelo final salvo em ppo-lunarlander.pt ✓")
    return agent, device

## Treinar o agente 🏃

A célula abaixo inicia o treino (os hiperparâmetros são os padrões do `parse_args` acima). Acompanhe pelo output: o **retorno dos episódios** deve subir ao longo do treino — um bom pouso chega a **+200** de recompensa.

In [ ]:
if __name__ == "__main__":
    args = parse_args()
    agent, device = main(args)

device: cuda
global_step=424, retorno_do_episódio=-171
Novo recorde! Modelo salvo em ppo-lunarlander.pt com retorno: -171 ✓
global_step=472, retorno_do_episódio=-320
global_step=476, retorno_do_episódio=-81
Novo recorde! Modelo salvo em ppo-lunarlander.pt com retorno: -81 ✓
global_step=484, retorno_do_episódio=-20
Novo recorde! Modelo salvo em ppo-lunarlander.pt com retorno: -20 ✓
update=1, SPS=1391
global_step=712, retorno_do_episódio=-103
global_step=776, retorno_do_episódio=-97
global_step=852, retorno_do_episódio=-97
global_step=948, retorno_do_episódio=-151
update=2, SPS=1414
global_step=1032, retorno_do_episódio=-101
global_step=1036, retorno_do_episódio=-255
global_step=1228, retorno_do_episódio=-122
global_step=1332, retorno_do_episódio=-143
global_step=1420, retorno_do_episódio=-384
global_step=1468, retorno_do_episódio=-165
global_step=1524, retorno_do_episódio=-149
update=3, SPS=1419
global_step=1632, retorno_do_episódio=-107
global_step=1672, retorno_do_episódio=-178
global

## Avaliar o agente 📈

In [ ]:
# Recria um único ambiente e carrega o modelo salvo
device = torch.device("cuda" if torch.cuda.is_available() and args.cuda else "cpu")
print("device:", device)
eval_envs = gym.vector.SyncVectorEnv([make_env(args.env_id, 0, 0)])
agent = Agent(eval_envs).to(device)
agent.load_state_dict(torch.load("ppo-lunarlander.pt", map_location=device))



n_eval_episodes = 10
episode_rewards = []
obs, _ = eval_envs.reset(seed=42)
for _ in range(n_eval_episodes):
    total = 0.0
    while True:
        with torch.no_grad():
            action, _, _, _ = agent.get_action_and_value(torch.Tensor(obs).to(device))
        obs, reward, terminations, truncations, info = eval_envs.step(action.cpu().numpy())
        total += reward[0]
        if terminations[0] or truncations[0]:
            break
    episode_rewards.append(total)
    obs, _ = eval_envs.reset()

print(f"Recompensa média = {np.mean(episode_rewards):.2f} +/- {np.std(episode_rewards):.2f}")

## Gravar um vídeo do replay 🎥

A função `record_video` cria um ambiente com `render_mode="rgb_array"` e grava o agente pousando a nave.

In [ ]:
import imageio
import os
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")  # necessário para renderizar no Colab

def record_video(env_id, agent, out_directory, fps=30, max_steps=1500):
    """
    Gera um vídeo de replay do agente treinado
    :param env_id: id do ambiente (ex.: "LunarLander-v3")
    :param agent: o agente PPO treinado
    :param out_directory: arquivo de saída (ex.: "replay-lunarlander.mp4")
    :param fps: quadros por segundo no vídeo
    :param max_steps: número máximo de passos a gravar
    """
    env = gym.make(env_id, render_mode="rgb_array")

    images = []
    state, _ = env.reset(seed=42)
    images.append(env.render())

    for _ in range(max_steps):
        with torch.no_grad():
            action, _, _, _ = agent.get_action_and_value(torch.Tensor(state).unsqueeze(0).to(device))
        state, reward, terminated, truncated, info = env.step(action[0].cpu().numpy())
        images.append(env.render())
        if terminated or truncated:
            break

    env.close()
    imageio.mimsave(out_directory, [np.array(img) for img in images], fps=fps)
    print(f"Vídeo salvo em {out_directory} ({len(images)} quadros)")

In [ ]:
# Gravar um vídeo do resultado do treino
video_path = "replay-lunarlander2.mp4"
record_video(args.env_id, agent, video_path)

## Alguns desafios adicionais 🏆

A melhor maneira de aprender **é tentar as coisas por conta própria**!

- **CartPole-v1**: troque `--env-id` para `CartPole-v1` — o mesmo código resolve em poucos passos.
- **MountainCar-v0** e **Acrobot-v1**: outros ambientes clássicos discretos que funcionam com esta implementação.
- **Hiperparâmetros**: experimente `--clip-coef` (0.1 a 0.3), `--update-epochs` e `--num-envs`. O que muda na velocidade e na estabilidade do treino?
- Compare com as métricas do TensorBoard (pasta `runs/`).

*(O código original do CleanRL também suporta ações contínuas (distribuição Normal); esta versão simplificada usa apenas ações discretas.)*

________________________________________________________________________
Parabéns por terminar esta atividade! 🥳

### Continue Aprendendo, Continue Incrível 🤗